# Conversation Memory in Modern LangChain with Groq

This notebook uses the current LangChain memory pattern:

```text
ChatPromptTemplate → ChatGroq
          ↑
RunnableWithMessageHistory → session-specific chat history
```

You will learn full buffer memory, session isolation, recent-window context, and direct message-history management.

> Select the **Python 3.13.7 (`.venv`)** kernel and run cells from top to bottom.

## 1. Configuration

Legacy tutorials often use `LLMChain`, `ConversationChain`, and classes from `langchain.memory`. Modern LangChain uses ordinary LCEL chains wrapped by `RunnableWithMessageHistory`.

```text
GROQ_API_KEY=your_key
GROQ_MODEL=qwen/qwen3.8-27b
GROQ_MAX_TOKENS=256
GROQ_TIMEOUT_SECONDS=30
```

In [ ]:
import logging
import os
from dataclasses import dataclass

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=True)
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(name)s | %(message)s", force=True)
logger = logging.getLogger("conversation_memory")


def positive_int(name: str, default: int) -> int:
    raw = os.getenv(name, str(default))
    try:
        value = int(raw)
    except ValueError as error:
        raise ValueError(f"{name} must be an integer, received {raw!r}.") from error
    if value <= 0:
        raise ValueError(f"{name} must be greater than zero.")
    return value


@dataclass(frozen=True, slots=True)
class Settings:
    model: str
    max_tokens: int
    timeout: int

    @classmethod
    def from_environment(cls) -> "Settings":
        if not os.getenv("GROQ_API_KEY"):
            raise RuntimeError("GROQ_API_KEY is missing from .env.")
        return cls(
            model=os.getenv("GROQ_MODEL", "qwen/qwen3.8-27b"),
            max_tokens=positive_int("GROQ_MAX_TOKENS", 256),
            timeout=positive_int("GROQ_TIMEOUT_SECONDS", 30),
        )


settings = Settings.from_environment()

In [ ]:
from langchain_groq import ChatGroq

chat_model = ChatGroq(
    model=settings.model,
    temperature=0.2,
    reasoning_effort="none",
    max_tokens=settings.max_tokens,
    timeout=settings.timeout,
    max_retries=2,
)

## 2. What does conversation memory mean?

Chat models do not automatically remember earlier API calls. An application must resend relevant previous messages with each new request.

| Term | Meaning |
| --- | --- |
| Message history | Ordered user and assistant messages |
| Session ID | Identifier that keeps one user's conversation separate |
| Memory store | Place where histories are saved |
| Context window | Messages actually sent to the model for the current request |

Memory does not change the model itself. It changes the messages included in the next prompt.

## 3. Full buffer memory with `RunnableWithMessageHistory`

### What?

`RunnableWithMessageHistory` loads a session's history, injects it into the chain, runs the model, and saves the new user/assistant messages.

### When and why?

Use it for multi-turn conversations. Always supply a session ID so different users do not share history.

The in-memory dictionary below is suitable for learning and tests. It is not durable and is not safe as a multi-process production store.

In [ ]:
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

conversation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Use conversation history when relevant. "
            "Do not claim to remember information that is absent from the supplied history.",
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),
    ]
)

base_conversation_chain = conversation_prompt | chat_model

history_store: dict[str, InMemoryChatMessageHistory] = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    clean_session_id = session_id.strip()
    if not clean_session_id:
        raise ValueError("session_id cannot be empty.")
    if clean_session_id not in history_store:
        history_store[clean_session_id] = InMemoryChatMessageHistory()
        logger.info("Created message history for session %s", clean_session_id)
    return history_store[clean_session_id]


conversation = RunnableWithMessageHistory(
    base_conversation_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

## 4. Invoke the same session multiple times

The configuration path `configurable.session_id` tells the wrapper which history to use.

The sample details below are fictional lesson data. Avoid storing sensitive information in chat history unless storage, retention, encryption, and access policies explicitly allow it.

In [ ]:
session_config = {"configurable": {"session_id": "student-demo-001"}}


def chat(question: str, *, config: dict) -> str:
    clean_question = question.strip()
    if not clean_question:
        raise ValueError("Question cannot be empty.")
    response = conversation.invoke({"question": clean_question}, config=config)
    return response.content


print(chat("Hello! My name is Sam.", config=session_config))
print(chat("I have moved 33 times.", config=session_config))
print(chat("If each move averaged 100 miles, what is the total distance?", config=session_config))
print(chat("What name did I tell you?", config=session_config))

## 5. Inspect message history

History contains typed messages such as `HumanMessage` and `AIMessage`, not plain strings. Typed roles prevent user text from being confused with system instructions.

In [ ]:
session_history = get_session_history("student-demo-001")

for index, message in enumerate(session_history.messages, start=1):
    print(f"{index}. {message.type}: {message.content}")

## 6. Session isolation

Each conversation must use its own session ID. This prevents one user's context from leaking into another user's responses.

In a real service, derive the session identity from authenticated server-side context rather than accepting an arbitrary client-provided user ID.

In [ ]:
second_session = {"configurable": {"session_id": "student-demo-002"}}

print(chat("Do you know my name?", config=second_session))
print("First session messages:", len(get_session_history("student-demo-001").messages))
print("Second session messages:", len(get_session_history("student-demo-002").messages))

## 7. Windowed context

### What?

A window keeps only the most recent messages in the context sent to the model.

### When and why?

Full histories grow continuously, increasing tokens, latency, and cost. A recent-message window provides bounded prompt growth.

This example retains the full in-memory audit history but sends only the last six prior messages to the model. Production systems often combine recent messages with a verified summary or retrieval over older conversation data.

In [ ]:
from typing import Any

from langchain_core.runnables import RunnableLambda

MAX_HISTORY_MESSAGES = 6


def keep_recent_history(data: dict[str, Any]) -> dict[str, Any]:
    history = list(data.get("chat_history", []))
    return {**data, "chat_history": history[-MAX_HISTORY_MESSAGES:]}


windowed_base_chain = (
    RunnableLambda(keep_recent_history)
    | conversation_prompt
    | chat_model
)

window_history_store: dict[str, InMemoryChatMessageHistory] = {}


def get_window_history(session_id: str) -> BaseChatMessageHistory:
    if not session_id.strip():
        raise ValueError("session_id cannot be empty.")
    return window_history_store.setdefault(session_id, InMemoryChatMessageHistory())


windowed_conversation = RunnableWithMessageHistory(
    windowed_base_chain,
    get_window_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

In [ ]:
window_config = {"configurable": {"session_id": "window-demo-001"}}
window_questions = [
    "My favorite color is blue.",
    "My favorite animal is a dog.",
    "I enjoy riding a scooter.",
    "My favorite city is San Francisco.",
    "What is my favorite city?",
]

for question in window_questions:
    response = windowed_conversation.invoke({"question": question}, config=window_config)
    print(f"User: {question}\nAssistant: {response.content}\n")

print("Messages retained in the store:", len(get_window_history("window-demo-001").messages))
print("Maximum prior messages sent per request:", MAX_HISTORY_MESSAGES)

## 8. Manage message history directly

Use `InMemoryChatMessageHistory` directly when you need to seed, inspect, clear, import, or export a conversation. Application code should usually interact through an abstraction so storage can later move to Redis, a database, or another durable service.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage

manual_history = InMemoryChatMessageHistory()
manual_history.add_messages(
    [
        HumanMessage(content="Hello!"),
        AIMessage(content="Hello! How can I help?"),
    ]
)

for message in manual_history.messages:
    print(f"{message.type}: {message.content}")

manual_history.clear()
print("Messages after clear():", len(manual_history.messages))

## 9. Choosing a memory strategy

| Strategy | Best for | Main risk |
| --- | --- | --- |
| Full buffer | Short conversations and debugging | Unbounded token growth |
| Recent window | Long chats needing recent context | Older facts disappear from model context |
| Summary + window | Long-running conversations | Summary errors can accumulate |
| Retrieval over history | Large histories and targeted recall | More infrastructure and retrieval evaluation |

## Enterprise checklist

- Isolate histories using authenticated tenant, user, and conversation identifiers.
- Use a durable store with encryption, access controls, expiry, and deletion support.
- Never log secrets or unrestricted message contents.
- Define retention and consent policies for personal information.
- Bound the context sent to the model.
- Handle concurrent writes and idempotent retries.
- Treat stored messages as untrusted input when placing them back into prompts.
- Test session isolation and memory behavior before deployment.